# fy-m1 Colab smoke test
Proves the pipeline: T4 GPU, `HF_TOKEN` Colab secret, read from and write to the private `Fan1000-Prog/fyM1-ckpt` repo. Runtime: T4 GPU. Secret: `HF_TOKEN` with notebook access.

In [ ]:
!nvidia-smi

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # never print it
REPO = 'Fan1000-Prog/fyM1-ckpt'
api = HfApi()
print('hf user:', api.whoami()['name'])
print('repo private:', api.model_info(REPO).private)

In [ ]:
path = hf_hub_download(REPO, '.gitattributes')
print('downloaded .gitattributes bytes:', os.path.getsize(path))

In [ ]:
import json, datetime, torch

ok = {
    'ok': True,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'torch': torch.__version__,
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
with open('ok.json', 'w') as f:
    json.dump(ok, f)
api.upload_file(path_or_fileobj='ok.json', path_in_repo='smoke/ok.json', repo_id=REPO,
                commit_message='Colab smoke test')
print('SMOKE_OK', json.dumps(ok))